# Data Cleaning

This notebook loads the raw tables, inspects them for quality issues, applies cleaning steps, and writes the results out as cleaned csv files.

In [1]:
import pandas as pd

## 1. Load Raw Data

In [2]:
INPUT_DATA_PATH = 'data/input/PART2-26-02-13_reorg'
OUTPUT_DATA_PATH = 'data/output'

train_hai = pd.read_csv(INPUT_DATA_PATH + '/train_hai.tsv', sep='\t')
train_participants = pd.read_csv(INPUT_DATA_PATH + '/train_participants.tsv', sep='\t')

In [3]:
name_df = {
    'train_hai': train_hai,
    'train_participants': train_participants,
}

## 2. Inspect Raw Data

In [4]:
for name, df in name_df.items():
    print(f"\n{'=' * 50}")
    print(f'TABLE: {name}  —  {df.shape[0]} rows, {df.shape[1]} columns')
    print(f"{'=' * 50}")
    print(df.dtypes, '\n')
    print('Missing values (%):')
    print((df.isna().mean() * 100).round(2))
    display(df.head(5))


TABLE: train_hai  —  128177 rows, 6 columns
hai_id                str
participant_id        str
timepoint         float64
virus_strain          str
value                 str
material              str
dtype: object 

Missing values (%):
hai_id            0.00
participant_id    0.00
timepoint         1.89
virus_strain      0.00
value             0.02
material          0.00
dtype: float64


,hai_id,participant_id,timepoint,virus_strain,value,material
0,ID_001__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_001,0.0,H1N1 A/South Carolina/1/1918,20.,Unknown
1,ID_001__2016_UGA_Standard_Fluzone__21__HAI__H1...,2016_UGA.ID_001,28.0,H1N1 A/South Carolina/1/1918,40.,Unknown
2,ID_002__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_002,0.0,H1N1 A/South Carolina/1/1918,5.,Unknown
3,ID_002__2016_UGA_Standard_Fluzone__21__HAI__H1...,2016_UGA.ID_002,28.0,H1N1 A/South Carolina/1/1918,5.,Unknown
4,ID_003__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_003,0.0,H1N1 A/South Carolina/1/1918,80.,Unknown



TABLE: train_participants  —  3960 rows, 17 columns
participant_id               str
subject                      str
biological_sex               str
race                         str
min_age                    int64
max_age                    int64
geolocation                  str
investigation_id             str
investigation_name           str
arm_id                       str
arm_name                     str
data_source                  str
description                  str
basic_curation               str
pubmed_ids                   str
main_pmid                    str
main_publication_author      str
dtype: object 

Missing values (%):
participant_id              0.00
subject                     0.00
biological_sex              0.00
race                        0.00
min_age                     0.00
max_age                     0.00
geolocation                20.18
investigation_id            0.00
investigation_name          0.00
arm_id                      0.00
arm_name            

,participant_id,subject,biological_sex,race,min_age,max_age,geolocation,investigation_id,investigation_name,arm_id,arm_name,data_source,description,basic_curation,pubmed_ids,main_pmid,main_publication_author
0,SDY269.SUB112836,SUB112836,female,White,28,28,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)
1,SDY269.SUB112849,SUB112849,female,Black or African American,39,39,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)
2,SDY269.SUB112854,SUB112854,male,Black or African American,46,46,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)
3,SDY269.SUB112860,SUB112860,female,White,32,32,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)
4,SDY269.SUB112881,SUB112881,female,Black or African American,29,29,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)


## 3. Clean HAI Table

Three issues to fix:
1. **Missing `timepoint` (~1.9%) and `value` (~0.02%)**. These rows carry no
   usable measurement, so we drop them.
2. **Non-numeric `value` entries**. Coerce to numeric; anything that cannot
   convert becomes NaN and is dropped.
3. **Placeholder strain entries (`'-'`)**. Not real strains; remove them.

In [5]:
print(f'Rows before cleaning: {len(train_hai)}')

# Drop rows where timepoint or value is missing
train_hai = train_hai.dropna(subset=['timepoint', 'value'])

# Coerce value to numeric; non-convertible entries become NaN
train_hai['value'] = pd.to_numeric(train_hai['value'], errors='coerce')
train_hai = train_hai.dropna(subset=['value'])

# Remove placeholder strain entries
train_hai = train_hai[train_hai['virus_strain'] != '-']

print(f'Rows after cleaning:  {len(train_hai)}')

Rows before cleaning: 128177
Rows after cleaning:  119840


## 4. Clean Participants Table

Three issues to fix:
1. **`main_publication_author` missing ~48%** — too sparse to be useful; drop
   the column.
2. **Inconsistent `biological_sex` casing** — normalise to lowercase.
3. **Noisy `race` labels** — collapse `'race: unknown'` and `'race: other'`
   into a single `'Unknown'` category.

In [6]:
print(f'Rows before cleaning: {len(train_participants)}')

# Drop the sparse publication column
train_participants = train_participants.drop(columns=['main_publication_author'])

# Normalise biological_sex
train_participants['biological_sex'] = (
    train_participants['biological_sex'].str.lower().str.strip()
)

# Standardize race labels
train_participants['race'] = train_participants['race'].replace({
    'race: unknown': 'Unknown',
    'race: other': 'Unknown',
})

print(f'Rows after cleaning:  {len(train_participants)}')

Rows before cleaning: 3960
Rows after cleaning:  3960


## 5. Export Cleaned Data

In [7]:
train_hai.to_csv(OUTPUT_DATA_PATH + '/train_hai_cleaned.csv', index=False)
train_participants.to_csv(OUTPUT_DATA_PATH + '/train_participants_cleaned.csv', index=False)

print('Wrote train_hai_cleaned.csv         ', train_hai.shape)
print('Wrote train_participants_cleaned.csv ', train_participants.shape)

Wrote train_hai_cleaned.csv          (119840, 6)
Wrote train_participants_cleaned.csv  (3960, 16)
